### 구조화된 출력(Structured Output) 완전정복 
#### GPT-4o-mini & EXAONE 3.5

LLM에게 자유롭게 글을 쓰게 하면 편하지만, 그 결과를 우리 시스템(DB, API, UI)에\
바로 꽂아 넣으려면 얘기가 달라집니다. \
"적당히 JSON처럼 보이는 텍스트"와 "우리 스키마에 100% 맞는 데이터"는 완전히 다른 신뢰도를 가집니다.

**OpenAI SDK와 Ollama 라이브러리를 직접 호출**해서 \
구조화된 출력을 얻는 방법을 가능한 많은 경우의 수로 다룹니다.

| 모델 | 역할 |
|---|---|
| `gpt-4o-mini` (OpenAI) | 클라우드 API 기반 구조화된 출력 (Structured Outputs 정식 지원) |
| `exaone3.5` (Ollama, 로컬) | 로컬 실행 모델의 구조화된 출력 (JSON Schema `format` 파라미터 방식) |

### 이 노트북에서 다루는 케이스 (총 14가지)

| # | 케이스 | 핵심 개념 |
|---|---|---|
| A | 레거시 JSON 모드 | `response_format={"type":"json_object"}` — 문법만 보장 |
| B | 수동 JSON Schema | dict로 직접 스키마 작성, `strict:true` |
| C | Pydantic + `.parse()` | 실무에서 가장 많이 쓰는 권장 방식 |
| D | 중첩 구조 | 리스트 안에 객체 (성적표) |
| E | Enum 분류 | 감성 분석 (긍정/부정/중립) |
| F | Optional/Nullable | 없을 수도 있는 필드 처리 |
| G | Union(anyOf) | 서로 다른 스키마 중 하나를 판별 |
| H | Refusal 처리 | 모델이 응답을 거부하는 경우 방어 코드 |
| I | 스트리밍 | 구조화된 출력 + 스트리밍 결합 |
| J | Function/Tool Calling | 함수 호출로 구조화된 값 얻기 |
| K | Ollama 네이티브 | `format=schema` 로컬 모델 구조화된 출력 |
| L | Ollama OpenAI 호환 | 같은 OpenAI 코드로 로컬 모델 접근 |
| M | 검증-재시도 패턴 | 실패를 감지하고 스스로 고쳐 재요청 |
| N | 두 모델 비교 | 같은 입력, 같은 스키마로 나란히 비교 |




### 1. 이론 — 구조화된 출력은 왜 필요하고, 어떻게 진화했나

#### 문제의 시작: "그냥 JSON으로 답해줘"
프롬프트에 "JSON 형식으로 답변해줘"라고 적어서 응답을 받으면, 다음과 같은 사고가 종종 발생합니다.

- 코드블록(` ```json ... ``` `)으로 감싸서 반환 → `json.loads()`가 실패
- 필드 이름이 매번 미묘하게 다름 (`"rating"` vs `"score"`)
- 필수 필드를 빼먹거나, 숫자를 문자열로 반환 (`"rating": "5"`)

이 문제는 "모델이 얼마나 똑똑한가"가 아니라 **"출력이 특정 문법/스키마를 따르도록 강제할 수 있는가"** 의\
문제입니다. 이를 해결하는 방법은 3단계로 발전해왔습니다.

#### 1단계 → 2단계 → 3단계

| 단계 | 방식 | 보장 수준 |
|---|---|---|
| 1단계 | 프롬프트로 부탁 ("JSON으로 답해줘") | 아무것도 보장 안 됨 |
| 2단계 | **JSON 모드** (`response_format={"type":"json_object"}`) | 문법적으로 유효한 JSON은 보장 (필드 구성은 미보장) |
| 3단계 | **JSON Schema / Structured Outputs** (`response_format={"type":"json_schema",...}`) | 스키마(필드명, 타입, 필수 여부)까지 보장 |

3단계는 "제약된 디코딩(Constrained Decoding)"이라는 기법으로 동작합니다.\
모델이 다음 토큰을 고를 때, 애초에 스키마를 위반하는 토큰은 후보에서 제외해버리는 방식입니다.\
그래서 "혹시 스키마를 어기면 어떡하지"라는 걱정 없이, \
나온 결과를 바로 `json.loads()` + Pydantic 검증에 넘길 수 있습니다.

#### 대안 경로: Function(Tool) Calling
"도구를 호출하게 만들고, 그 도구의 인자(arguments)를 구조화된 데이터로 받는" 방식도 있습니다.\
원래는 실제 함수를 실행시키기 위한 기능이지만, **호출을 강제(`tool_choice`)** 하면\
"함수를 부르는 시늉만 하고 인자값만 구조화된 데이터로 받아내는" 용도로도 활용됩니다.\
이 노트북의 Case J에서 다룹니다.

#### 로컬 모델(EXAONE 등)은 다르다
OpenAI의 Structured Outputs는 OpenAI 서버 안에서 동작하는 전용 기능입니다.\
로컬에서 Ollama로 돌리는 EXAONE 3.5 같은 모델은 **Ollama가 자체적으로 제공하는\
`format` 파라미터**(llama.cpp의 GBNF 문법 제약 기능을 사용)로 유사한 효과를 냅니다.\
API 형태는 다르지만 "스키마를 주면 그 스키마를 따르는 JSON만 나온다"는 목표는 같습니다.


### 2. 환경 설정


In [ ]:
# 우리는 설치되어 있음
# %pip install -q openai ollama pydantic

In [1]:
import os
import json
from enum import Enum
from typing import Optional, Union, Literal

from pydantic import BaseModel, ValidationError
import openai
from openai import OpenAI
import ollama

# --- OpenAI 설정 ---
# 터미널에서 미리 설정해두세요.
#   (Windows PowerShell) $env:OPENAI_API_KEY = "sk-..."
#   (macOS/Linux)        export OPENAI_API_KEY="sk-..."
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = "gpt-4o-mini"
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# --- Ollama 설정 ---
OLLAMA_MODEL = "exaone3.5"


def need_openai() -> bool:
    """OPENAI_API_KEY가 없으면 셀 실행을 건너뛰고 안내만 출력합니다."""
    if not OPENAI_API_KEY:
        print("[건너뜀] OPENAI_API_KEY가 설정되지 않아 이 셀은 실행하지 않습니다.")
        return False
    return True


def need_ollama() -> bool:
    """Ollama 서버에 연결할 수 없으면 셀 실행을 건너뛰고 안내만 출력합니다."""
    try:
        ollama.list()
        return True
    except Exception as e:
        print(f"[건너뜀] Ollama 서버에 연결할 수 없습니다 ({e}).")
        print(f"        'ollama serve' 실행 및 'ollama pull {OLLAMA_MODEL}' 확인 후 다시 실행하세요.")
        return False


print("OpenAI API 키:", "감지됨" if OPENAI_API_KEY else "없음 -> OpenAI 예제는 건너뜁니다")
print("Ollama 연결:", "가능" if need_ollama() else "불가 -> Ollama 예제는 건너뜁니다")


OpenAI API 키: 감지됨
Ollama 연결: 가능


---
### Case A — 레거시 JSON 모드 (`json_object`)

가장 오래된 방식입니다. **문법적으로 유효한 JSON**이라는 것만 보장하고,\
필드 구성이 스키마와 일치하는지는 보장하지 않습니다.\
프롬프트에 원하는 필드 이름을 직접 명시해야 하고, 그마저도 모델이 빠뜨릴 수 있습니다.


In [2]:
if need_openai():
    prompt = (
        "다음 리뷰에서 상품명과 평점(1~5 정수)을 JSON으로 추출해줘. "
        '형식: {"product": "...", "rating": ...}\n\n'
        "리뷰: 노트북 정말 좋아요! 배터리도 오래가고 화면도 선명해요. 5점 만점에 5점 줍니다."
    )
    completion = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "사용자가 요청한 정보를 JSON으로만 응답하세요."},
            {"role": "user", "content": prompt},
        ],
    )
    raw = completion.choices[0].message.content
    print("원본 응답 문자열:", raw)
    data = json.loads(raw)  # 문법은 항상 보장되므로 파싱 자체는 실패하지 않음
    print("파싱 결과:", data)
    print("주의: 'product', 'rating' 필드가 실제로 존재하는지는 우리가 직접 확인해야 합니다.")


원본 응답 문자열: {"product": "노트북", "rating": 5}
파싱 결과: {'product': '노트북', 'rating': 5}
주의: 'product', 'rating' 필드가 실제로 존재하는지는 우리가 직접 확인해야 합니다.


---
### Case B — 수동으로 작성한 JSON Schema (Strict 모드)

Pydantic 없이, **JSON Schema를 dict로 직접 작성**해서 `response_format`에 넣는 방식입니다.\
이 방식을 이해해두면, Pydantic이 내부적으로 무엇을 자동 생성해주는지 감을 잡을 수 있습니다.\

Strict 모드의 핵심 제약 3가지:\
- 모든 필드는 `required`에 포함되어야 함 (없어도 되는 필드는 "F. Optional" 케이스에서처럼 null 허용 타입으로 표현)
- 객체마다 `"additionalProperties": false` 필요 (스키마에 없는 필드 생성 방지)
- 최상위는 반드시 `"type": "object"`


In [3]:
#응답 받고 싶은 형식을 json schema로 정의하고, openai에 전달하면, openai가 응답을 생성할 때, json schema를 준수하도록 강제할 수 있음
if need_openai():
    manual_schema = {
        "type": "object",
        "properties": { # 반드시 있어야 되는 필수 필드
            "product": {"type": "string"},
            "rating": {"type": "integer"},
        },
        "required": ["product", "rating"],
        "additionalProperties": False,
    }
    completion = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "review_extraction",
                "schema": manual_schema,
                "strict": True,
            },
        },
        messages=[ 
            {"role": "system", "content": "리뷰에서 상품명과 평점을 추출하세요."},
            {"role": "user", "content": "노트북 정말 좋아요! 5점 만점에 5점 줍니다."},
        ],
    )
    raw = completion.choices[0].message.content
    print("원본 응답:", raw)
    print("파싱 결과:", json.loads(raw))
    print("-> 이번엔 'product'/'rating' 필드가 반드시 존재함이 보장됩니다.")


원본 응답: {"product":"노트북","rating":5}
파싱 결과: {'product': '노트북', 'rating': 5}
-> 이번엔 'product'/'rating' 필드가 반드시 존재함이 보장됩니다.


---
### Case C — Pydantic + `.parse()` (실무 권장 방식)

매번 JSON Schema를 dict로 직접 작성하는 건 번거롭고 실수하기 쉽습니다.\
OpenAI 파이썬 SDK는 **Pydantic 모델을 그대로 넘기면 스키마 생성 + 파싱까지 자동으로 처리**해줍니다.\

`client.beta.chat.completions.parse(response_format=PydanticModel클래스)` 를 쓰면\
`completion.choices[0].message.parsed` 에 **타입이 있는 파이썬 객체**가 바로 담깁니다.\
이후 케이스들은 대부분 이 방식을 기본으로 사용합니다.


In [4]:
class ReviewExtraction(BaseModel):
    product: str
    rating: int


if need_openai():
    completion = openai_client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "리뷰에서 상품명과 평점을 추출하세요."},
            {"role": "user", "content": "노트북 정말 좋아요! 5점 만점에 5점 줍니다."},
        ],
        response_format=ReviewExtraction,
    )
    result = completion.choices[0].message.parsed
    print("타입:", type(result))
    print("객체:", result)
    print("상품명:", result.product, "| 평점:", result.rating)


타입: <class '__main__.ReviewExtraction'>
객체: product='노트북' rating=5
상품명: 노트북 | 평점: 5


---
### Case D — 중첩 구조 (리스트 안에 객체)

실무 데이터는 대부분 계층 구조를 가집니다. 학생 한 명의 성적표처럼\
**객체 안에 리스트, 리스트 안에 또 다른 객체**가 들어가는 경우를 다뤄봅니다.\

> 참고: OpenAI Structured Outputs는 스키마 하나당 **객체의 속성 100개, 중첩 깊이 5단계**까지\
> 지원합니다. 너무 깊은 구조는 평평하게(flat) 재설계하는 것이 안정적입니다.


In [5]:
class Subject(BaseModel):
    name: str
    score: int

class StudentReport(BaseModel):
    student_name: str
    grade_level: int
    subjects: list[Subject]
    average_score: float

if need_openai():
    text = "김민준 학생은 3학년이고, 국어 88점, 수학 95점, 영어 76점을 받았습니다."
    completion = openai_client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "학생 성적 정보를 구조화해서 추출하세요. "
                    "average_score는 과목 점수들의 평균을 직접 계산해서 채우세요."
                ),
            },
            {"role": "user", "content": text},
        ],
        response_format=StudentReport,
    )
    report = completion.choices[0].message.parsed
    print(f"{report.student_name} ({report.grade_level}학년) - 평균 {report.average_score}점")
    for s in report.subjects:
        print(f"  - {s.name}: {s.score}점")


김민준 (3학년) - 평균 86.33333333333333점
  - 국어: 88점
  - 수학: 95점
  - 영어: 76점


---
### Case E — Enum 기반 분류 (감성 분석)

값이 정해진 몇 가지 카테고리 중 하나여야 한다면 자유 문자열보다 **Enum**을 쓰는 게 훨씬 안전합니다.\
Enum을 쓰면 모델이 "긍정적"처럼 스키마에 없는 변형된 값을 절대 만들어낼 수 없습니다.


In [8]:
class Sentiment(str, Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"

class SentimentResult(BaseModel):
    sentiment: Sentiment
    confidence: float
    reason: str

reviews = [
    "배송이 너무 느리고 포장도 엉망이었어요.",
    "그냥 무난했어요. 특별한 건 없네요.",
    "가격 대비 최고의 제품입니다!",
]

if need_openai():
    for review in reviews:
        completion = openai_client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "리뷰의 감성을 분석하세요."},
                {"role": "user", "content": review},
            ],
            response_format=SentimentResult,
        )
        r = completion.choices[0].message.parsed
        print(f"[{r.sentiment.value}] (확신도 {r.confidence:.2f}) {review}")
        print(f"    -> {r.reason}")


[부정] (확신도 0.95) 배송이 너무 느리고 포장도 엉망이었어요.
    -> 배송 지연과 포장 불량에 대한 불만이 표현되어 있어 부정적인 감정을 나타냄.
[중립] (확신도 0.80) 그냥 무난했어요. 특별한 건 없네요.
    -> 리뷰에서 '무난했다'는 표현은 긍정도 부정도 아닌 중립적인 감정을 나타내며, 특별한 점이 없다는 언급은 감정이 더 체계적이지 않음을 의미합니다.
[긍정] (확신도 0.95) 가격 대비 최고의 제품입니다!
    -> 제품에 대한 높은 만족도를 표현하고 있으며, 가격과 품질의 비율이 좋다는 긍정적인 피드백이 포함되어 있습니다.


---
### Case F — Optional(Nullable) 필드

"있을 수도, 없을 수도 있는" 필드는 어떻게 다룰까?\
**`Optional[str] = None`** 으로 선언하면 Pydantic/OpenAI SDK가 자동으로\
`anyOf: [string, null]` 타입으로 변환해줍니다.

중요한 점: strict 모드에서는 **모든 필드가 항상 `required`** 이므로, "없으면 아예 필드를 만들지 않는다"가\
아니라 "필드는 항상 있지만 값이 `null`일 수 있다"가 됩니다. 이 차이를 헷갈리지 않는 것이 중요합니다.


In [7]:
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None  # 없으면 모델이 null을 채워 넣음

texts = [
    "저는 둘리이고 이메일은 dooly@example.com 입니다.",
    "도우너입니다. 이메일은 dauner@example.com, 전화번호는 010-1234-5678이에요.",
]

if need_openai():
    for t in texts:
        completion = openai_client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "연락처 정보를 추출하세요. 전화번호가 텍스트에 없으면 null로 두세요."},
                {"role": "user", "content": t},
            ],
            response_format=ContactInfo,
        )
        print(completion.choices[0].message.parsed)


name='둘리' email='dooly@example.com' phone=None
name='도우너' email='dauner@example.com' phone='010-1234-5678'


---
### Case G — Union(anyOf): 서로 다른 스키마 중 하나 판별하기

"이 일정이 회의인지 휴가인지 모르지만, 둘 중 하나의 스키마로 구조화하고 싶다"는 상황입니다.\
`Literal["meeting"]` / `Literal["vacation"]` 처럼 **판별 필드(discriminator)** 를 각 하위 모델에 넣어두면,\
나중에 `entry.event_type` 값만 보고도 어떤 타입인지 안전하게 분기할 수 있습니다.


In [ ]:
class MeetingEvent(BaseModel):
    event_type: Literal["meeting"]
    title: str
    attendees: list[str]


class VacationEvent(BaseModel):
    event_type: Literal["vacation"]
    person: str
    days: int


class CalendarEntry(BaseModel):
    entry: Union[MeetingEvent, VacationEvent]


texts = [
    "다음 주 월요일에 둘리, 또치와 프로젝트 회의가 있어요.",
    "민지가 3일간 휴가를 냈습니다.",
]

if need_openai():
    for t in texts:
        completion = openai_client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "일정 텍스트를 회의 또는 휴가 이벤트로 분류해서 구조화하세요."},
                {"role": "user", "content": t},
            ],
            response_format=CalendarEntry,
        )
        entry = completion.choices[0].message.parsed.entry
        if isinstance(entry, MeetingEvent):
            print(f"[회의] {entry.title} (참석자: {', '.join(entry.attendees)})")
        else:
            print(f"[휴가] {entry.person} - {entry.days}일")


---
### Case H — Refusal(응답 거부) 처리

Structured Outputs 사용 중 모델이 안전상의 이유로 응답을 거부하면,\
스키마를 채우는 대신 `message.refusal` 필드에 거부 사유가 채워집니다.\
**`.parsed`를 확인하기 전에 항상 `.refusal`을 먼저 체크**하는 방어적인 코드 패턴을 익혀둡니다.\
(아래 예제 자체는 무해한 리뷰라 실제로 거부되지는 않지만, 프로덕션 코드에서는 이 체크가 필수입니다.)


In [11]:
if need_openai():
    completion = openai_client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "리뷰의 감성을 분석하세요."},
            {"role": "user", "content": "이 카페 분위기가 정말 좋아요."},
        ],
        response_format=SentimentResult,
    )
    msg = completion.choices[0].message

    # 실무 코드에서 항상 넣어야 하는 방어 패턴
    if msg.refusal:
        print("모델이 응답을 거부했습니다:", msg.refusal)
    else:
        print("정상 파싱 결과:", msg.parsed)


정상 파싱 결과: sentiment=<Sentiment.POSITIVE: '긍정'> confidence=0.95 reason='카페의 분위기를 긍정적으로 표현하고 있습니다.'


---
### Case I — 스트리밍 + 구조화된 출력

구조화된 출력도 스트리밍이 가능합니다. `client.beta.chat.completions.stream()` 컨텍스트\
매니저를 사용하면, 토큰이 오는 대로 받아보다가 마지막에 `get_final_completion()`으로\
완성된(그리고 파싱된) 결과를 얻을 수 있습니다. 긴 응답을 UI에 점진적으로 보여줄 때 유용합니다.


In [ ]:
if need_openai(): #Execution Context
    with openai_client.beta.chat.completions.stream( #with는 실행 컨텍스트가 아니라 컨텍스트 매니저
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "리뷰의 감성을 분석하세요."},
            {"role": "user", "content": "배송도 빠르고 품질도 훌륭해요. 재구매 의사 100%입니다."},
        ],
        response_format=SentimentResult,
    ) as stream: 
        for event in stream:
            if event.type == "content.delta":
                print(event.delta, end="", flush=True)
        final = stream.get_final_completion()

    print()  # 줄바꿈
    print("최종 파싱 결과:", final.choices[0].message.parsed)


---
### Case J — Function(Tool) Calling으로 구조화된 출력 얻기

원래 Tool Calling은 "모델이 실제 함수를 실행시키기 위해" 만들어졌지만,\
`tool_choice`로 **특정 함수 호출을 강제**하면 "함수의 인자값"이라는 형태로\
구조화된 데이터를 받아낼 수 있습니다. `response_format`을 쓰는 방식과 결과적으로 유사하지만,\
"이 모델의 다른 실제 도구들과 함께 섞어 써야 하는" 에이전트 구조에서는 이 방식이 더 자연스러울 때가 있습니다.\

`openai.pydantic_function_tool()` 헬퍼가 Pydantic 모델을 도구 스펙으로 변환해줍니다.


In [12]:
sentiment_tool = openai.pydantic_function_tool(
    SentimentResult, 
    name="save_sentiment",
    description="분석된 감성 결과를 저장합니다.",
)

if need_openai():
    completion = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "리뷰를 분석하고 save_sentiment 함수를 호출해 결과를 저장하세요."},
            {"role": "user", "content": "직원분이 친절했지만 음식은 그저 그랬어요."},
        ],
        tools=[sentiment_tool],
        tool_choice={"type": "function", "function": {"name": "save_sentiment"}},  # 무조건 이 함수를 호출하도록 강제
    )
    tool_call = completion.choices[0].message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    result = SentimentResult.model_validate(args)
    print("호출된 함수:", tool_call.function.name)
    print("구조화된 인자:", result)

호출된 함수: save_sentiment
구조화된 인자: sentiment=<Sentiment.NEUTRAL: '중립'> confidence=0.8 reason='직원에 대한 긍정적인 평가와 음식에 대한 중립적인 평가가 혼합되어 있습니다.'


In [10]:
sentiment_tool

{'type': 'function',
 'function': {'name': 'save_sentiment',
  'strict': True,
  'parameters': {'$defs': {'Sentiment': {'enum': ['긍정', '부정', '중립'],
     'title': 'Sentiment',
     'type': 'string'}},
   'properties': {'sentiment': {'$ref': '#/$defs/Sentiment'},
    'confidence': {'title': 'Confidence', 'type': 'number'},
    'reason': {'title': 'Reason', 'type': 'string'}},
   'required': ['sentiment', 'confidence', 'reason'],
   'title': 'SentimentResult',
   'type': 'object',
   'additionalProperties': False},
  'description': '분석된 감성 결과를 저장합니다.'}}

---
### 3. 이론 — 로컬 모델(EXAONE 3.5)의 구조화된 출력은 무엇이 다른가

OpenAI의 Structured Outputs는 OpenAI 서버 내부에서만 동작하는 전용 기능입니다.\
Ollama로 로컬에서 돌리는 EXAONE 3.5는 **Ollama 자체의 `format` 파라미터**를 사용합니다.

| | OpenAI (`gpt-4o-mini`) | Ollama (`exaone3.5`) |
|---|---|---|
| 스키마 전달 위치 | `response_format={"type":"json_schema", "json_schema": {...}}` | `format=<JSON Schema dict>` |
| Pydantic 연동 | `response_format=PydanticModel` (SDK가 자동 변환+파싱) | `format=Model.model_json_schema()` 로 직접 전달, 응답은 `Model.model_validate_json()`으로 직접 파싱 |
| 신뢰도 | 100%에 가까움 (OpenAI 자체 벤치마크 기준) | 모델/양자화 수준에 따라 달라짐 — 검증 로직이 더 중요해짐 |
| 스트리밍/거부(refusal) 개념 | 지원 | 없음 (로컬 모델은 안전 거부 매커니즘이 별도로 없음) |

즉, **로컬 모델을 쓸수록 Case M의 "검증-재시도 패턴"이 훨씬 중요해집니다.**\
OpenAI는 스키마를 어길 수가 없지만, 로컬 모델은 (특히 파라미터 수가 작을수록) 가끔\
JSON을 깨뜨리거나 타입을 잘못 채울 수 있기 때문입니다.


---
### Case K — Ollama 네이티브 방식 (EXAONE 3.5)

`ollama.chat()`의 `format` 파라미터에 `Model.model_json_schema()`를 그대로 넘기면 됩니다.\
응답은 `response.message.content`에 JSON **문자열**로 들어오므로,\
`Model.model_validate_json()`으로 직접 파싱+검증합니다.\

In [ ]:
if need_ollama():
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": "다음 리뷰의 감성을 분석해줘: '가격 대비 훌륭한 성능입니다.' JSON으로만 답변해줘.",
            }
        ],
        format=SentimentResult.model_json_schema(),
        options={"temperature": 0},  # 낮은 온도가 스키마 준수율을 높여줍니다.
    )
    result = SentimentResult.model_validate_json(response.message.content)
    print(result)


---
### Case L — Ollama의 OpenAI 호환 엔드포인트 사용하기

Ollama는 `http://localhost:11434/v1` 에 **OpenAI와 동일한 REST 형태**의 엔드포인트도 제공합니다.\
즉, `base_url`만 바꾸면 **OpenAI용으로 짠 코드를 거의 그대로 재사용**할 수 있습니다.

다만 `response_format={"type":"json_schema",...}` 지원은 Ollama 버전에 따라 동작이 달라질 수 있어서,\
**신뢰도가 중요한 실습에서는 Case K(네이티브 방식)를 기본으로 추천**합니다.\
이 방식은 "여러 LLM 제공자를 같은 인터페이스로 다루고 싶을 때"의 참고용으로 알아두면 좋습니다.


In [ ]:
ollama_openai_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # 키는 무시되지만 형식상 필요

if need_ollama():
    completion = ollama_openai_client.chat.completions.create(
        model=OLLAMA_MODEL,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "sentiment_result",
                "schema": SentimentResult.model_json_schema(),
                "strict": True,
            },
        },
        messages=[
            {"role": "user", "content": "다음 리뷰의 감성을 분석해줘: '배송이 너무 늦었어요.'"}
        ],
    )
    raw = completion.choices[0].message.content
    print("원본 응답:", raw)
    print("파싱 결과:", SentimentResult.model_validate_json(raw))


---
### Case M — 검증-재시도(Validate-Retry) 패턴

Structured Outputs가 없는 환경(구형 모델, 일부 로컬 모델)이나 Ollama 호환 레이어가\
기대만큼 스키마를 지키지 않는 경우를 대비해, **직접 검증하고 실패하면 스스로 고쳐서\
재요청하는 유틸리티**를 만들어봅니다.

이 부분은 실제 LLM 호출 없이도 동작을 100% 확인할 수 있도록, 아래 셀에서\
**"불안정한 모델"을 흉내내는 목업(mock) 함수**로 실제 재시도 로직을 직접 실행해봅니다.\
(1차 시도: JSON 문법 오류 → 2차 시도: 스키마 위반 → 3차 시도: 정상)


In [ ]:
#형식은 맞는데, 응답이 맘에 안든다면 이런식으로 할 수 있따. 
class Person(BaseModel):
    name: str
    age: int


# 실제 LLM 대신, "불안정한 모델의 응답"을 순서대로 흉내내는 목업 함수
_mock_attempts = [
    '{"name": "홍길동", "age": "서른",}',   # 1차: JSON 문법 오류 (trailing comma) + 타입 오류
    '{"name": "홍길동", "age": "서른"}',    # 2차: 문법은 OK, 타입 오류 (age가 문자열)
    '{"name": "홍길동", "age": 30}',        # 3차: 정상
]
_attempt_state = {"n": 0}


def mock_llm_call(prompt: str) -> str:
    """실제 LLM 호출 대신 위 시나리오를 순서대로 반환합니다."""
    idx = min(_attempt_state["n"], len(_mock_attempts) - 1)
    _attempt_state["n"] += 1
    return _mock_attempts[idx]


def call_with_retry(schema_model, llm_call_fn, prompt: str, max_retries: int = 3):
    """LLM 호출 -> JSON 파싱 -> 스키마 검증을 시도하고, 실패하면 오류 내용을 프롬프트에
    덧붙여 재요청하는 범용 유틸리티. 실제 서비스에서는 llm_call_fn 자리에 진짜
    openai_client.chat.completions.create(...) 또는 ollama.chat(...) 호출을 넣으면 됩니다.
    """
    last_error = None
    current_prompt = prompt

    for attempt in range(1, max_retries + 1):
        raw = llm_call_fn(current_prompt)

        try:
            parsed_json = json.loads(raw)
        except json.JSONDecodeError as e:
            last_error = f"JSON 문법 오류: {e}"
            print(f"[시도 {attempt}] JSON 파싱 실패 -> {last_error}")
            current_prompt = (
                f"{prompt}\n\n[이전 응답에 문제가 있었습니다] {last_error}\n"
                f"문제가 된 응답: {raw}\n올바른 JSON 문법으로 다시 응답해줘."
            )
            continue

        try:
            result = schema_model.model_validate(parsed_json)
            print(f"[시도 {attempt}] 검증 성공!")
            return result
        except ValidationError as e:
            last_error = str(e)
            print(f"[시도 {attempt}] 스키마 검증 실패")
            current_prompt = (
                f"{prompt}\n\n[이전 응답에 문제가 있었습니다] 스키마 검증 실패: {last_error}\n"
                f"문제가 된 응답: {raw}\n스키마에 맞게 다시 응답해줘."
            )

    raise RuntimeError(f"최대 재시도 횟수({max_retries}) 초과. 마지막 오류: {last_error}")


result = call_with_retry(Person, mock_llm_call, "사람 정보를 추출해줘")
print()
print("최종 결과:", result)


---
### Case N — GPT-4o-mini vs EXAONE 3.5 나란히 비교

같은 스키마(`SentimentResult`), 같은 입력 리뷰를 두 모델에 동시에 던져서\
결과를 나란히 비교해봅니다. 클라우드 모델과 로컬 모델의 감성 분석 성향 차이를\
직접 눈으로 확인할 수 있는 실습입니다.


In [ ]:
if need_openai() and need_ollama():
    review = "생각보다 화면이 크고 선명해서 만족스러워요. 다만 배터리가 반나절 밖에 안 가는건 아쉽네요."

    gpt_completion = openai_client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "리뷰의 감성을 분석하세요."},
            {"role": "user", "content": review},
        ],
        response_format=SentimentResult,
    )
    gpt_result = gpt_completion.choices[0].message.parsed

    ollama_response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": f"다음 리뷰의 감성을 분석해줘: \'{review}\'"}],
        format=SentimentResult.model_json_schema(),
        options={"temperature": 0},
    )
    ollama_result = SentimentResult.model_validate_json(ollama_response.message.content)

    print("리뷰:", review)
    print()
    print("GPT-4o-mini ->", gpt_result)
    print("EXAONE 3.5  ->", ollama_result)


---
### 4. 참고자료 — OpenAI Structured Outputs 제약사항 정리

실제로 스키마를 설계할 때 자주 걸리는 제약사항들입니다. (2026년 기준, 변경될 수 있으니
최종 확인은 [OpenAI 공식 문서](https://platform.openai.com/docs/guides/structured-outputs)를 참고하세요.)

| 제약 | 내용 |
|---|---|
| 필수 필드 | 모든 필드가 `required`에 있어야 함. "선택적" 필드는 `anyOf: [type, null]` 로 표현 (Optional + 기본값 None) |
| additionalProperties | 모든 객체에 `"additionalProperties": false` 필요 (Pydantic 사용 시 SDK가 자동 처리) |
| 중첩 깊이 | 최대 5단계 |
| 객체 속성 개수 | 스키마 전체에서 최대 100개 |
| 문자열 총 길이 | 속성명 + enum 값 + const 값의 총 길이 15,000자 이내 |
| enum 값 개수 | 전체 최대 500개, 250개 초과 시 총 문자열 길이 7,500자 이내 |
| 미지원 키워드 | `minLength`, `maxLength`, `pattern`, `minimum`, `maximum` 등은 무시됨 (모델이 강제로 지키지 않음) |
| 재귀 스키마 | Pydantic으로 자기 자신을 참조하는 모델(트리 구조 등)은 SDK 자동 변환에서 오류 발생 → 깊이를 고정하거나 ID로 참조하는 평평한 구조로 재설계 필요 |

### 방식 선택 가이드

- **정형화된 데이터 추출/분류가 목적** → Case C (Pydantic + `.parse()`) 를 기본값으로 사용
- **다른 실제 도구(Tool)들과 함께 에이전트를 구성** → Case J (Function Calling) 가 더 자연스러움
- **긴 응답을 점진적으로 UI에 보여줘야 함** → Case I (스트리밍)
- **로컬/오픈소스 모델 사용** → Case K를 기본으로, 반드시 Case M(검증-재시도)을 함께 적용


---
## 정리 — 실무 체크리스트

1. **가능하면 Pydantic + `.parse()`를 기본으로 사용**하세요. 스키마 dict를 직접 짜는 것보다 실수가 적습니다.
2. **"없을 수도 있는 값"은 `Optional[X] = None`으로 명시**하세요. 그냥 타입 힌트를 생략하면 안 됩니다.
3. **분류 문제는 자유 문자열 대신 반드시 Enum/Literal을 쓰세요.** 스키마에 없는 값이 나올 여지를 원천 차단합니다.
4. **`message.refusal` 체크를 습관화하세요.** 특히 사용자 입력을 그대로 모델에 넘기는 서비스라면 필수입니다.
5. **로컬 모델(Ollama)을 쓸수록 검증-재시도 로직이 필수**입니다. OpenAI만큼의 100% 보장이 없습니다.
6. **스키마는 최대한 평평하게** 설계하세요. 중첩이 깊어질수록 지연시간도 늘고, 모델이 실수할 여지도 늘어납니다.
7. **Structured Outputs는 "형식"만 보장합니다. "내용"이 맞는지는 여전히 우리 책임**입니다.
   (예: `invoice_total` 필드가 스키마는 맞지만 계산이 틀릴 수 있음 — 검증 로직/휴먼 리뷰는 별개로 필요)
